# <b style='font-size:30px;font-family:Arial'>Create a File-Content-Based Collection using JSON Files</b>

File-content-based vector collections are ideal for unstructured data where you provide files as the input source. The workflow is:
1. Configure the Ingestor pipeline with `.files()` and `.upsert()`
2. Execute the pipeline with `.run()` to create collection, ingest files, generate embeddings, and create index
3. Perform similarity searches

**Key Learning:** For FILE-CONTENT-BASED collections, use the method chain `.files().upsert(embedding_model=<model>).run()` where embeddings are generated from the text field during ingestion.

## <b style='font-size:20px;font-family:Arial'>1. Import Required Libraries</b>

In [ ]:
import sys
import os
import json
import glob
import getpass
from datetime import datetime

# TeradataML imports
from teradataml import create_context, remove_context, DataFrame

# Teradata Vector Store V2 API - Ingestor workflow
from teradatagenai import Collection, CollectionManager, ColumnInfo
from teradatagenai import BasicIngestor, ExtractionSchema
from teradatagenai import LocalConfig, S3Config, AzureBlobConfig
from teradatagenai import TeradataAI
from teradatagenai.vector_store.Ingestor import Ingestor
from teradatagenai.common.constants import CollectionType
from teradatasqlalchemy.types import VARCHAR, CLOB

print("✅ All imports successful!")
print("✓ Using Vector Store V2 API - Ingestor stepwise pipeline")
print("✓ Collection Type: FILE-CONTENT-BASED")

✅ All imports successful!
✓ Using Vector Store V2 API - Ingestor stepwise pipeline
✓ Collection Type: FILE-CONTENT-BASED


## <b style='font-size:20px;font-family:Arial'>2. Connect to Teradata Vantage</b>

In [ ]:
# Configure connection parameters
os.environ['TD_HOST'] = getpass.getpass('Enter Teradata Host: ')
os.environ['TD_USERNAME'] = getpass.getpass('Enter Teradata Username: ')
os.environ['TD_PASSWORD'] = getpass.getpass('Enter Teradata Password: ')
os.environ['TD_BASE_URL'] = getpass.getpass('Enter Base URL: ')
os.environ['TD_AUTH_MECH'] = 'BASIC'
os.environ['NVIDIA_API_KEY'] = getpass.getpass('Enter NVIDIA API Key: ')

# Create connection context 
con = create_context()
print(f"✓ Connected to Teradata Vantage at {os.environ['TD_HOST']}")

# Verify service health
try:
    health = CollectionManager.health()
    print("✓ Vector Store service is healthy")
except Exception as e:
    print(f"Service status: {e}")

Authentication token is generated and set for the session.


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatasqlalchemy\telemetry\queryband.py:557: UserWarning: [Teradata][teradataml](TDML_2002) Overwriting an existing context associated with Teradata Vantage Connection. Most of the operations on any teradataml DataFrames created before this will not work.
  return exposed_func(*args, **kwargs)


Authentication token is generated and set for the session.
✓ Connected to Teradata Vantage at 10.27.178.11
✓ Vector Store service is healthy


## <b style='font-size:20px;font-family:Arial'>3. Configure Embedding Model</b>

Specify the embedding model that will generate vector embeddings from the text content.

In [ ]:
# Configure embedding model (runtime generation)
os.environ["NVIDIA_API_KEY"] = getpass.getpass('Enter NVIDIA API Key: ')
embedding_model = TeradataAI(
    api_type="nim",
    model_name=getpass.getpass('Enter Embedding Model Name: '),
    api_base=getpass.getpass('Enter Embedding Model API Base URL: ')
)

print(f"✓ Embedding Model: {embedding_model.model_name}")
print("✓ Embeddings will be generated automatically during ingestion")

✓ Embedding Model: nvidia/llama-3.2-nv-embedqa-1b-v2
✓ Embeddings will be generated automatically during ingestion


## <b style='font-size:20px;font-family:Arial'>4. Prepare Json Input Data</b>

Load Json files for the vector collection. You can load from local storage, S3, or Azure Blob Storage.

## <b style='font-size:20px;font-family:Arial'>4a. Load from Local Storage</b>

Using Local files for ingestion

In [5]:
# Use existing test data directory (cross-platform compatible path)
notebook_dir = os.getcwd()
data_dir = os.path.join(os.path.dirname(notebook_dir), "example-data", "json_files")
json_file_path = os.path.join(data_dir, "documents_combined_from_individual.json")

# Verify file exists
if os.path.exists(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        documents = json.load(f)
    print(f"✓ Loaded {len(documents)} documents from {os.path.basename(json_file_path)}")
    print(f"✓ File location: {data_dir}")
else:
    print(f"⚠ File not found: {json_file_path}")
    print("Creating sample data...")

# Configure local file source
local_config = LocalConfig(
    files=[json_file_path],
    files_type="json"
)
print(f"\n✓ LocalConfig created for local file ingestion")

✓ Loaded 4 documents from documents_combined_from_individual.json
✓ File location: c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\notebooks\example-data\json_files

✓ LocalConfig created for local file ingestion


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_local = f"json_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_local = ExtractionSchema(
    table_name=f"json_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor = (
    Ingestor(
        name=collection_name_local,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="oaf",
        description="File-Content-Based collection from JSON files"
    )
    .files(
        files=local_config,  # Uses storage_config from Section 3c
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_local
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_local}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(local_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: json_file_content_20260224_174558
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: LocalConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [7]:
# Check if collection already exists and destroy it
try:
    existing_collection_local = Collection(name=collection_name_local)
    if existing_collection_local.exists:
        print(f"⚠ Collection '{collection_name_local}' already exists. Destroying it first...")
        existing_collection_local.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)  # Wait for cleanup
except Exception as e:
    pass

# Execute ingestion (creates collection and ingests files)
print("Starting ingestion pipeline...")
result = ingestor.run()
print("✓ Ingestion completed - files uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/10

Perform similarity search and access the actual results from the _SimilaritySearch object

In [8]:
search_results = existing_collection_local.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"  # Returns _SimilaritySearch object with results in 'similar_objects' property
)

# 
results_list = search_results.similar_objects  # This is the actual list of results
result_count = search_results.similar_objects_count  # Count property

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...

3. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Text: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_local.get_indexes_embeddings()  # Get sample of ingested data with embeddings
    print(f"\n✓ Sample data from collection '{collection_name_local}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")

Destroy the collection

In [19]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_local.destroy()
print(f"✓ Collection '{collection_name_local}' destroyed")

Collection destroy request is accepted and in progress
✓ Collection 'json_file_content_20260224_082901' destroyed


## <b style='font-size:20px;font-family:Arial'>4b. Load from S3 Storage</b>

Instead of local files, you can ingest JSON files directly from Amazon S3.

In [ ]:
from teradatagenai.vector_store import S3Config

# S3 Configuration
s3_config = S3Config(
    bucket=getpass.getpass('Enter S3 Bucket: '),
    key=getpass.getpass('Enter S3 Key: '),
    aws_access_key_id=getpass.getpass('Enter AWS Access Key ID: '),
    aws_secret_access_key=getpass.getpass('Enter AWS Secret Access Key: '),
    region_name=getpass.getpass('Enter AWS Region: '),
    files_type="json",
    aws_session_token=getpass.getpass('Enter AWS Session Token: ')
)

print("✓ S3 storage configuration enabled")
print(f"  Bucket: {s3_config.bucket}")
print(f"  Region: {s3_config.region_name}")

✓ S3 storage configuration enabled
  Bucket: genaitestaanchal
  Region: us-west-2


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_s3 = f"json_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_s3 = ExtractionSchema(
    table_name=f"json_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor = (
    Ingestor(
        name=collection_name_s3,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="oaf",
        description="File-Content-Based collection from JSON files"
    )
    .files(
        files=s3_config,  # Uses s3_config from Section 3c
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_s3
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_s3}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(s3_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: json_file_content_20260216_204308
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: S3Config
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [12]:
# Check if collection already exists and destroy it
try:
    existing_collection_s3 = Collection(name=collection_name_s3)
    if existing_collection_s3.exists:
        print(f"⚠ Collection '{collection_name_s3}' already exists. Destroying it first...")
        existing_collection_s3.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)  # Wait for cleanup
except Exception as e:
    pass

# Execute ingestion (creates collection and ingests files)
print("Starting ingestion pipeline...")
result = ingestor.run()
print("✓ Ingestion completed - files uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'json_file_content_20260216_204308' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [13]:
# Perform similarity search
search_results = existing_collection_s3.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"  # Returns _SimilaritySearch object with results in 'similar_objects' property
)

# Access the actual results from the _SimilaritySearch object
results_list = search_results.similar_objects  # This is the actual list of results
result_count = search_results.similar_objects_count  # Count property

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...

3. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Text: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_s3.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_s3}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_json_file_content_20260216_204308_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
oaf,json_content_table_20260216204308,3,documents_combined_from_individual.json,"-0.026290999725461,-0.00044299999717623,0.0170140005648136,-0.0243989992886782,0.0368349999189377,0.000423999998020008,0.025237999856472,-0.0194850005209446,0.0320740006864071,0.0574040003120899,-0.0105969998985529,0.00300200004130602,-0.015433999709785,0.0232090000063181,0.00138799997512251,-0.0234990008175373,0.000627000001259148,0.0178989991545677,-0.0221560001373291,0.0418090000748634,-0.00228899996727705,-0.00579500012099743,-0.0475159995257854,-0.00349200004711747,0.0112380003556609,0.0104980003088713,0.0266880001872778,0.0130540002137423,-0.00612600008025765,0.00523399980738759,0.0155720002949238,-0.0258479993790388,-0.0078959995880723,-0.0274509992450476,-0.0123669998720288,-0.0111079998314381,-0.0179440006613731,0.010071000084281,0.00642400002107024,0.0353390015661716,0.0282899998128414,0.00417299987748265,0.0219570007175207,-0.00862099975347519,-0.00287600001320243,0.0147780003026128,0.00792699959129095,0.0188449993729591,-0.0118410000577569,-0.0134429996833205,-0.0116039998829365,0.0162509996443987","-0.0262969620525837,-0.000443100434495136,0.0170178581029177,-0.0244045313447714,0.0368433520197868,0.000424096127972007,0.0252437219023705,-0.0194894187152386,0.0320812724530697,0.0574170164763927,-0.0105994027107954,0.00300268083810806,-0.0154374996200204,0.0232142619788647,0.00138831464573741,-0.0235043298453093,0.000627142144367099,0.0179030578583479,-0.0221610236912966,0.041818480938673,-0.00228951894678175,-0.00579631421715021,-0.0475267730653286,-0.00349279190413654,0.0112405484542251,0.010500380769372,0.0266940519213676,0.0130569599568844,-0.00612738914787769,0.00523518677800894,0.0155755309388041,-0.0258538611233234,-0.00789778959006071,-0.0274572242051363,-0.012369804084301,-0.0111105181276798,-0.0179480686783791,0.0100732836872339,0.00642545660957694,0.0353470146656036,0.0282964147627354,0.00417394610121846,0.0219619795680046,-0.00862295459955931,-0.0028766521718353,0.0147813512012362,0.00792879704385996,0.0188492722809315,-0.0118436850607395,-0.0134460479021072,-0.0116066308692098,0.01625468395650"
oaf,json_content_table_20260216204308,2,documents_combined_from_individual.json,"0.0006520000169985,0.0181430000811815,0.0143659999594092,0.017272999510169,0.015022000297904,0.0484310016036034,0.00796500034630299,0.0117950001731515,0.00762199983000755,0.000608999980613589,-0.0298159997910261,0.00758400000631809,-0.00393299991264939,0.0375979989767075,-0.00171400001272559,0.00981099996715784,0.00519199995324016,-0.0361020006239414,-0.000555999984499067,0.00410099979490042,0.0217129997909069,-0.0280150007456541,0.00447800010442734,-0.0103230001404881,0.015777999535203,-0.00441000005230308,0.00744199985638261,-0.0192109998315573,0.0258789993822575,0.0215610004961491,0.0509340018033981,0.0341800004243851,-0.00476500019431114,-0.00890399981290102,-0.0199280008673668,-0.0214079990983009,-0.0281980000436306,-0.0396730005741119,0.00385099998675287,-0.0034099998883903,-0.0300600007176399,-0.01112399995327,0.00694999983534217,1.59999999596039e-05,-0.0138320000842214,-0.000264000002061948,-0.00534099992364645,0.0428470000624657,-0.00271199992857873,-0.0123519999906421,0.0153729999437928,-0.037597998","0.000652212474960834,0.0181489121168852,0.0143706817179918,0.01727862842381,0.0150268953293562,0.0484467819333076,0.00796759594231844,0.0117988437414169,0.00762448366731405,0.000609198410529643,-0.0298257153481245,0.00758647127076983,-0.00393428141251206,0.0376102514564991,-0.00171455857343972,0.0098141971975565,0.00519369170069695,-0.036113765090704,-0.000556181184947491,0.00410233624279499,0.0217200759798288,-0.0280241295695305,0.00447945948690176,-0.0103263640776277,0.0157831404358149,-0.00441143708303571,0.00744442502036691,-0.0192172601819038,0.025887431576848,0.021568026393652,0.0509505979716778,0.0341911390423775,-0.00476655270904303,-0.00890690088272095,-0.0199344

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
collection.destroy()
print(f"✓ Collection '{collection_name_blob}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4c. (Alternative) Load from Azure Blob Storage</b>

Instead of local files or S3, you can ingest JSON files from Azure Blob Storage.

In [ ]:
from teradatagenai.vector_store import AzureBlobConfig

# Azure Blob Configuration
azure_config = AzureBlobConfig(
    container=getpass.getpass('Enter Azure Container: '),
    blob_name=getpass.getpass('Enter Blob Name: '),
    account_name=getpass.getpass('Enter Azure Account Name: '),
    account_key=getpass.getpass('Enter Azure Account Key: '),
    files_type="json"
)

print("✓ Azure Blob storage configuration provided")
print(f"  Container: {azure_config.container}")
print(f"  Account: {azure_config.account_name}")

 Azure Blob storage configuration provided
  Container: genaitestcontainer
  Account: genaitestaanchal
  Note: account_key is empty - may require SAS token or managed identity


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_blob = f"json_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema = ExtractionSchema(
    table_name=f"json_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor = (
    Ingestor(
        name=collection_name_blob,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="oaf",
        description="File-Content-Based collection from JSON files"
    )
    .files(
        files=azure_config,  # Uses azure_config from Section 3c
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_blob}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(azure_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: json_file_content_20260216_205932
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: AzureBlobConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [17]:
# Check if collection already exists and destroy it
try:
    existing_collection_blob = Collection(name=collection_name_blob)
    if existing_collection_blob.exists:
        print(f"⚠ Collection '{collection_name_blob}' already exists. Destroying it first...")
        existing_collection_blob.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)  # Wait for cleanup
except Exception as e:
    pass

# Execute ingestion (creates collection and ingests files)
print("Starting ingestion pipeline...")
result = ingestor.run()
print("✓ Ingestion completed - files uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'json_file_content_20260216_205932' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [18]:
# Perform similarity search
search_results = existing_collection_blob.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"  # Returns _SimilaritySearch object with results in 'similar_objects' property
)

# Access the actual results from the _SimilaritySearch object
results_list = search_results.similar_objects  # This is the actual list of results
result_count = search_results.similar_objects_count  # Count property

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...

3. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Text: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_blob.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_blob}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_json_file_content_20260216_205932_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
oaf,json_content_table_20260216205932,2,documents_combined_from_individual.json,"0.0006520000169985,0.0181430000811815,0.0143659999594092,0.017272999510169,0.015022000297904,0.0484310016036034,0.00796500034630299,0.0117950001731515,0.00762199983000755,0.000608999980613589,-0.0298159997910261,0.00758400000631809,-0.00393299991264939,0.0375979989767075,-0.00171400001272559,0.00981099996715784,0.00519199995324016,-0.0361020006239414,-0.000555999984499067,0.00410099979490042,0.0217129997909069,-0.0280150007456541,0.00447800010442734,-0.0103230001404881,0.015777999535203,-0.00441000005230308,0.00744199985638261,-0.0192109998315573,0.0258789993822575,0.0215610004961491,0.0509340018033981,0.0341800004243851,-0.00476500019431114,-0.00890399981290102,-0.0199280008673668,-0.0214079990983009,-0.0281980000436306,-0.0396730005741119,0.00385099998675287,-0.0034099998883903,-0.0300600007176399,-0.01112399995327,0.00694999983534217,1.59999999596039e-05,-0.0138320000842214,-0.000264000002061948,-0.00534099992364645,0.0428470000624657,-0.00271199992857873,-0.0123519999906421,0.0153729999437928,-0.037597998","0.000652212474960834,0.0181489121168852,0.0143706817179918,0.01727862842381,0.0150268953293562,0.0484467819333076,0.00796759594231844,0.0117988437414169,0.00762448366731405,0.000609198410529643,-0.0298257153481245,0.00758647127076983,-0.00393428141251206,0.0376102514564991,-0.00171455857343972,0.0098141971975565,0.00519369170069695,-0.036113765090704,-0.000556181184947491,0.00410233624279499,0.0217200759798288,-0.0280241295695305,0.00447945948690176,-0.0103263640776277,0.0157831404358149,-0.00441143708303571,0.00744442502036691,-0.0192172601819038,0.025887431576848,0.021568026393652,0.0509505979716778,0.0341911390423775,-0.00476655270904303,-0.00890690088272095,-0.019934494048357,-0.0214149747043848,-0.0282071884721518,-0.0396859273314476,0.00385225494392216,-0.00341111118905246,-0.0300697963684797,-0.0111276246607304,0.00695226481184363,1.60052131832344e-05,-0.0138365076854825,-0.000264086032984778,-0.00534274056553841,0.0428609624505043,-0.00271288375370204,-0.0123560251668096,0.0153780095279217,-0.03761025"
oaf,json_content_table_20260216205932,3,documents_combined_from_individual.json,"-0.026290999725461,-0.00044299999717623,0.0170140005648136,-0.0243989992886782,0.0368349999189377,0.000423999998020008,0.025237999856472,-0.0194850005209446,0.0320740006864071,0.0574040003120899,-0.0105969998985529,0.00300200004130602,-0.015433999709785,0.0232090000063181,0.00138799997512251,-0.0234990008175373,0.000627000001259148,0.0178989991545677,-0.0221560001373291,0.0418090000748634,-0.00228899996727705,-0.00579500012099743,-0.0475159995257854,-0.00349200004711747,0.0112380003556609,0.0104980003088713,0.0266880001872778,0.0130540002137423,-0.00612600008025765,0.00523399980738759,0.0155720002949238,-0.0258479993790388,-0.0078959995880723,-0.0274509992450476,-0.0123669998720288,-0.0111079998314381,-0.0179440006613731,0.010071000084281,0.00642400002107024,0.0353390015661716,0.0282899998128414,0.00417299987748265,0.0219570007175207,-0.00862099975347519,-0.00287600001320243,0.0147780003026128,0.00792699959129095,0.0188449993729591,-0.0118410000577569,-0.0134429996833205,-0.0116039998829365,0.0162509996443987","-0.0262969620525837,-0.000443100434495136,0.0170178581029177,-0.0244045313447714,0.0368433520197868,0.000424096127972007,0.0252437219023705,-0.0194894187152386,0.0320812724530697,0.0574170164763927,-0.0105994027107954,0.00300268083810806,-0.0154374996200204,0.0232142619788647,0.00138831464573741,-0.0235043298453093,0.000627142144367099,0.0179030578583479,-0.0221610236912966,0.041818480938673,-0.00228951894678175,-0.00579631421715021,-0.0475267730653286,-0.00349279190413654,0.0112405484542251,0.010500380769372,0.0266940519213676,0.0130569599568844,-0.00612738914787769,0.00523518677800894,0.0155755309388041,-0.0258538611233234,-0.00789778959006071,-0.0274572242051363,-0.012

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
collection.destroy()
print(f"✓ Collection '{collection_name_s3}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4d. Load from Google Cloud Storage</b>

Instead of local files, S3, or Azure, you can ingest JSON files directly from Google Cloud Platform (GCP) Storage.

In [ ]:
from teradatagenai import GCPConfig

# GCP Configuration
gcp_config = GCPConfig(
    files_type="json",
    bucket=getpass.getpass('Enter GCP Bucket: '),
    blob_name=getpass.getpass('Enter GCP Blob Name: '),
    project_id=getpass.getpass('Enter GCP Project ID: '),
    secret=json.loads(getpass.getpass('Enter GCP Service Account JSON (paste entire JSON): '))
)

print("✓ GCP storage configuration enabled")
print(f"  Bucket: {gcp_config.bucket}")
print(f"  Blob: {gcp_config.blob_name}")
print(f"  Project: {gcp_config.project_id}")

✓ GCP storage configuration enabled
  Bucket: ak250090-filestore
  Blob: harsha_files/documents_combined_from_individual.json
  Project: icgcp-vector-store


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_gcp = f"json_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_gcp = ExtractionSchema(
    table_name=f"json_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor_gcp = (
    Ingestor(
        name=collection_name_gcp,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="oaf",
        description="File-Content-Based collection from JSON files with runtime embedding generation"
    )
    .files(
        files=gcp_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_gcp
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_gcp}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(gcp_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: json_file_content_20260216_211634
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: GCPConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [27]:
# Check if collection already exists and destroy it
try:
    existing_collection_gcp = Collection(name=collection_name_gcp)
    if existing_collection_gcp.exists:
        print(f"⚠ Collection '{collection_name_gcp}' already exists. Destroying it first...")
        existing_collection_gcp.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_gcp = ingestor_gcp.run()
print("✓ Ingestion completed - embeddings generated and stored")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'json_file_content_20260216_211634' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [28]:
# Perform similarity search
search_results_gcp = existing_collection_gcp.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_gcp.similar_objects
result_count = search_results_gcp.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...

3. Retrieval Augmented Generation (RAG)
   Author: Bob Wilson
   Category: ai
   Text: RAG combines large language models with external knowledge retrieval to generate more accurate and contextually relevant responses. The system retriev...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_gcp.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_gcp}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_json_file_content_20260216_211634_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
oaf,json_content_table_20260216211634,2,documents_combined_from_individual.json,"0.0006520000169985,0.0181430000811815,0.0143659999594092,0.017272999510169,0.015022000297904,0.0484310016036034,0.00796500034630299,0.0117950001731515,0.00762199983000755,0.000608999980613589,-0.0298159997910261,0.00758400000631809,-0.00393299991264939,0.0375979989767075,-0.00171400001272559,0.00981099996715784,0.00519199995324016,-0.0361020006239414,-0.000555999984499067,0.00410099979490042,0.0217129997909069,-0.0280150007456541,0.00447800010442734,-0.0103230001404881,0.015777999535203,-0.00441000005230308,0.00744199985638261,-0.0192109998315573,0.0258789993822575,0.0215610004961491,0.0509340018033981,0.0341800004243851,-0.00476500019431114,-0.00890399981290102,-0.0199280008673668,-0.0214079990983009,-0.0281980000436306,-0.0396730005741119,0.00385099998675287,-0.0034099998883903,-0.0300600007176399,-0.01112399995327,0.00694999983534217,1.59999999596039e-05,-0.0138320000842214,-0.000264000002061948,-0.00534099992364645,0.0428470000624657,-0.00271199992857873,-0.0123519999906421,0.0153729999437928,-0.037597998","0.000652212474960834,0.0181489121168852,0.0143706817179918,0.01727862842381,0.0150268953293562,0.0484467819333076,0.00796759594231844,0.0117988437414169,0.00762448366731405,0.000609198410529643,-0.0298257153481245,0.00758647127076983,-0.00393428141251206,0.0376102514564991,-0.00171455857343972,0.0098141971975565,0.00519369170069695,-0.036113765090704,-0.000556181184947491,0.00410233624279499,0.0217200759798288,-0.0280241295695305,0.00447945948690176,-0.0103263640776277,0.0157831404358149,-0.00441143708303571,0.00744442502036691,-0.0192172601819038,0.025887431576848,0.021568026393652,0.0509505979716778,0.0341911390423775,-0.00476655270904303,-0.00890690088272095,-0.019934494048357,-0.0214149747043848,-0.0282071884721518,-0.0396859273314476,0.00385225494392216,-0.00341111118905246,-0.0300697963684797,-0.0111276246607304,0.00695226481184363,1.60052131832344e-05,-0.0138365076854825,-0.000264086032984778,-0.00534274056553841,0.0428609624505043,-0.00271288375370204,-0.0123560251668096,0.0153780095279217,-0.03761025"
oaf,json_content_table_20260216211634,3,documents_combined_from_individual.json,"-0.026290999725461,-0.00044299999717623,0.0170140005648136,-0.0243989992886782,0.0368349999189377,0.000423999998020008,0.025237999856472,-0.0194850005209446,0.0320740006864071,0.0574040003120899,-0.0105969998985529,0.00300200004130602,-0.015433999709785,0.0232090000063181,0.00138799997512251,-0.0234990008175373,0.000627000001259148,0.0178989991545677,-0.0221560001373291,0.0418090000748634,-0.00228899996727705,-0.00579500012099743,-0.0475159995257854,-0.00349200004711747,0.0112380003556609,0.0104980003088713,0.0266880001872778,0.0130540002137423,-0.00612600008025765,0.00523399980738759,0.0155720002949238,-0.0258479993790388,-0.0078959995880723,-0.0274509992450476,-0.0123669998720288,-0.0111079998314381,-0.0179440006613731,0.010071000084281,0.00642400002107024,0.0353390015661716,0.0282899998128414,0.00417299987748265,0.0219570007175207,-0.00862099975347519,-0.00287600001320243,0.0147780003026128,0.00792699959129095,0.0188449993729591,-0.0118410000577569,-0.0134429996833205,-0.0116039998829365,0.0162509996443987","-0.0262969620525837,-0.000443100434495136,0.0170178581029177,-0.0244045313447714,0.0368433520197868,0.000424096127972007,0.0252437219023705,-0.0194894187152386,0.0320812724530697,0.0574170164763927,-0.0105994027107954,0.00300268083810806,-0.0154374996200204,0.0232142619788647,0.00138831464573741,-0.0235043298453093,0.000627142144367099,0.0179030578583479,-0.0221610236912966,0.041818480938673,-0.00228951894678175,-0.00579631421715021,-0.0475267730653286,-0.00349279190413654,0.0112405484542251,0.010500380769372,0.0266940519213676,0.0130569599568844,-0.00612738914787769,0.00523518677800894,0.0155755309388041,-0.0258538611233234,-0.00789778959006071,-0.0274572242051363,-0.012

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_gcp.destroy()
print(f"✓ Collection '{collection_name_gcp}' destroyed")

---

## Summary

This notebook demonstrated how to:
- Create a FILE-CONTENT-BASED vector collection from JSON files
- Configure the Ingestor pipeline with extraction schema and embedding model
- Test with four different storage types: Local, S3, Azure Blob, and GCP
- Generate embeddings automatically during ingestion
- Perform semantic similarity search on the collection

**Key Points:**
- JSON files must be in **array format**: `[{...}, {...}]`
- Embeddings are generated automatically from the `text` field using the specified embedding model
- Metadata columns are preserved for filtering and retrieval
- Each storage type (Local/S3/Azure/GCP) has its own complete isolated workflow
- The collection supports semantic search out of the box